# drone3d — quality-max reconstruction on Kaggle
**Before running (Settings panel):** Accelerator = **GPU T4 ×2** (or P100 — both 16 GB), **Internet = ON**, **Persistence = ON**. **Not TPU** (our stack is CUDA/PyTorch). The notebook uses **one** 16 GB GPU (the 2nd T4 sits idle); MCMC caps keep VRAM bounded so it won't OOM-crash mid-run.
Everything is fetched from public sources; nothing is uploaded. Use *Save & Run All (Commit)* for 9 h background runs.

## 1. Clone the repo

In [ ]:
import os, subprocess, glob, json
os.chdir('/kaggle/working')
if not os.path.isdir('drone3d'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/sank6112/drone3d'], check=True)
else:  # persistence ON -> make sure we're on the latest master, not stale code
    subprocess.run(['git','-C','drone3d','pull','--ff-only'])
os.chdir('/kaggle/working/drone3d'); print('cwd:', os.getcwd())
subprocess.run(['git','log','--oneline','-1'])

## 2. Environment + data setup
Set `RUBBLE=1` to also pull Mill-19 Rubble (9.2 GB, the big aerial scene).

In [ ]:
os.environ['RUBBLE']='0'   # '1' to also fetch Rubble
os.environ['MAST3R']='1'
!bash scripts/setup_kaggle.sh

## 2b. Smoke test (~2 min) — confirm GPU + deps before the long runs
Runs 500 steps on Sculpture at low res. If a `.splat` appears, the environment is good.

In [ ]:
subprocess.run(['bash','scripts/train_quality.sh',
                'data/dronesplat/Sculpture','4','outputs/_smoke','300000','500',
                '--save-steps','500','--eval-steps','500'])
ok = os.path.exists('outputs/_smoke/model.splat')
print('SMOKE TEST', 'PASSED ✅' if ok else 'FAILED ❌ — check the log above')

## 3. Ablation on Sculpture — which quality knobs actually help?
Short 7k runs at factor 2 so we can compare knobs fast, then lock the winner.

In [ ]:
ABL = {
  'A0_baseline':            [],
  'A1_aa':                  ['--antialiased'],
  'A2_aa_app':              ['--antialiased','--app_opt'],
  'A3_aa_app_bilat':        ['--antialiased','--app_opt','--use_bilateral_grid'],
  'A4_aa_app_pose':         ['--antialiased','--app_opt','--pose_opt'],
}
for name, flags in ABL.items():
    rd = f'outputs/abl_{name}'
    subprocess.run(['bash','scripts/train_quality.sh',
                    'data/dronesplat/Sculpture','2',rd,'1500000','7000',*flags])

### Ablation results

In [ ]:
rows=[]
for d in sorted(glob.glob('outputs/abl_*')):
    js=sorted(glob.glob(d+'/stats/val_step*.json'))
    if not js: continue
    v=json.load(open(js[-1]))
    rows.append((d.split('abl_')[1], v['psnr'], v['ssim'], v['lpips'], v['num_GS']))
print(f"{'config':24s}{'PSNR':>8}{'SSIM':>8}{'LPIPS':>8}{'#GS':>11}")
for r in rows: print(f'{r[0]:24s}{r[1]:8.2f}{r[2]:8.3f}{r[3]:8.3f}{r[4]:11,}')
if rows:
    best=min(rows,key=lambda r:r[3]); print('\nBEST by LPIPS:', best[0])

## 4. Final full-res Sculpture (factor 1, 30k) with the best knobs
Edit `BEST_FLAGS` from the ablation above if a different combo won.

In [ ]:
BEST_FLAGS=['--antialiased','--app_opt','--pose_opt']
subprocess.run(['bash','scripts/train_quality.sh',
                'data/dronesplat/Sculpture','1','outputs/sculpture_final','2000000','30000',*BEST_FLAGS])

## 5. Product path — MASt3R poses from raw images (no COLMAP)
Kaggle's 32 GB RAM allows more frames than the 6 GB laptop; `--pose_opt` refines the MASt3R poses.

In [ ]:
subprocess.run(['python','scripts/mast3r_to_colmap.py',
                '--frames','data/dronesplat/Sculpture/images',
                '--out','outputs/sculpture_mast3r_colmap','--max-frames','20'])
subprocess.run(['bash','scripts/train_quality.sh',
                'outputs/sculpture_mast3r_colmap','1','outputs/sculpture_mast3r_final',
                '1500000','30000','--antialiased','--app_opt','--pose_opt'])

## 6. Rubble — big aerial scene (needs RUBBLE=1 in step 2)

In [ ]:
if os.path.isdir('data/rubble/rubble-pixsfm/train/rgbs'):
    subprocess.run(['python','scripts/meganerf_to_colmap.py',
                    '--src','data/rubble/rubble-pixsfm/train','--out','data/rubble_sub','--n','500','--long','1024'])
    subprocess.run(['bash','scripts/train_quality.sh',
                    'data/rubble_sub','1','outputs/rubble_final','2000000','40000',
                    '--antialiased','--app_opt','--use_bilateral_grid'])
else:
    print('Rubble not present — set RUBBLE=1 in step 2 and re-run setup.')

## 7. Collect deliverables
`.splat` files (drag into supersplat.com) + a metrics summary. Download from the Output panel.

In [ ]:
os.makedirs('/kaggle/working/deliverables', exist_ok=True)
for f in glob.glob('outputs/*/model.splat'):
    dst='/kaggle/working/deliverables/'+f.split('/')[-2]+'.splat'
    subprocess.run(['cp',f,dst])
subprocess.run(['python','scripts/write_summary.py'])
print(open('outputs/SUMMARY.md').read())
print('\nDeliverables:'); print('\n'.join(sorted(glob.glob('/kaggle/working/deliverables/*'))))